# Allomorph: Initial Structure Generation Examples

This notebook demonstrates how to use the `allomorph` library to generate monometallic, bimetallic, and trimetallic nanoparticles with various shapes and chemical orderings.

## 1. Custom Configuration

First, we define our custom elements and update the global configuration.

In [2]:
from allomorph.constants import ELE_DICT, update_constants, validate_ele_dict
from allomorph.init_struct.gen_bnp_al import gen_bnp
from allomorph.init_struct.gen_bnp_cs import gen_hard_core_shell
from allomorph.init_struct.gen_mnp import gen_mnp
from allomorph.init_struct.gen_tnp_al import gen_tnp

# Define custom elements
custom_ele_dict = {
    "Cu": {"lc": {"FCC": 3.61}, "radius": 1.28, "mass": 63.55,
           "rho": 8960, "m": 0.063546, "bulkE": 3.49},
    "Ag": {"lc": {"FCC": 4.09}, "radius": 1.44, "mass": 107.87,
           "rho": 10490, "m": 0.107868, "bulkE": 2.95},
    "Au": {"lc": {"FCC": 4.09}, "radius": 1.44, "mass": 196.97,
           "rho": 19320, "m": 0.196967, "bulkE": 3.81},
}

# Validate and update constants
validate_ele_dict(custom_ele_dict)
update_constants({"ELE_DICT": custom_ele_dict, "RANDOM_DISTRIB_NO": 2})

print("Configuration updated.")

Configuration updated.


## 2. Monometallic Nanoparticles (MNP)

We can generate nanoparticles with various shapes (e.g., Octahedron 'OT', Truncated Octahedron 'TO', Icosahedron 'IC', Decahedron 'DH') and diameters.

In [3]:
# Generate an octahedral Cu nanoparticle with diameter 30 Angstrom
cu_ot = gen_mnp(
    shape='OT',
    diameter=30,
    element='Cu',
    latConst=ELE_DICT['Cu']['lc']['FCC']
)
print(f"Cu Octahedron (30A): {len(cu_ot)} atoms")

# Generate an icosahedral Ag nanoparticle with diameter 20 Angstrom
ag_ic = gen_mnp(
    shape='IC',
    diameter=20,
    element='Ag',
    latConst=ELE_DICT['Ag']['lc']['FCC']
)
print(f"Ag Icosahedron (20A): {len(ag_ic)} atoms")

Cu Octahedron (30A): 489 atoms
Ag Icosahedron (20A): 309 atoms


## 3. Bimetallic Nanoparticles (BNP)

BNPs can be generated by modifying an existing MNP or by combining two MNPs.

In [4]:
print("--- Randomly Distributed Alloy (RAL) ---")
# Convert 30% of Cu atoms in the octahedron to Ag randomly
cu_ag_ral = gen_bnp(
    obj=cu_ot.copy(),
    element1='Cu',
    element2='Ag',
    shape='OT',
    ratio2=30,
    distrib='RAL',
    rseed=42
)
print(f"Cu-Ag RAL: {cu_ag_ral.get_chemical_formula()}")

print("\n--- Randomly Distributed Core-Shell-like Alloy (RCS) ---")
cu_ag_rcs = gen_bnp(
    obj=cu_ot.copy(),
    element1='Cu',
    element2='Ag',
    shape='OT',
    ratio2=50,
    distrib='RCS',
    rseed=42
)
print(f"Cu-Ag RCS: {cu_ag_rcs.get_chemical_formula()}")

print("\n--- Hard Core-Shell (M1@M2) ---")
# Combine a 20A Ag core with the outer part of a 30A Cu shell
cu_ag_cs = gen_hard_core_shell(
    shell_atoms=cu_ot.copy(),
    core_atoms=ag_ic.copy(),
    del_cutoff=(ELE_DICT['Cu']['radius'] + ELE_DICT['Ag']['radius']) / 2
)
print(f"Ag@Cu Core-Shell: {cu_ag_cs.get_chemical_formula()}")

--- Randomly Distributed Alloy (RAL) ---
Cu-Ag RAL: Ag147Cu342

--- Randomly Distributed Core-Shell-like Alloy (RCS) ---
Cu-Ag RCS: Ag244Cu245

--- Hard Core-Shell (M1@M2) ---
Ag@Cu Core-Shell: Ag309Cu240


## 4. Trimetallic Nanoparticles (TNP)

TNPs are generated by further modifying a bimetallic structure or using specific trimetallic distributions.

In [5]:
print("--- Trimetallic Random Alloy (RAL + RAL) ---")
cu_ag_au_tnp = gen_tnp(
    obj=cu_ag_ral.copy(),
    element1='Cu',
    element2='Ag',
    element3='Au',
    ele1Ratio=40,
    ele2Ratio=30,
    ele3Ratio=30,
    distrib1='RAL',
    distrib2='RAL',
    rseed=0
)
print(f"Cu-Ag-Au TNP: {cu_ag_au_tnp.get_chemical_formula()}")

--- Trimetallic Random Alloy (RAL + RAL) ---
Cu-Ag-Au TNP: Ag147Au146Cu196


## 5. Visualization

You can visualize the generated structures using ASE's GUI (requires a display).

In [ ]:
from ase.visualize import view

view(cu_ag_au_tnp)

<Popen: returncode: None args: ['/home/DART/jonathant/Documents/PhD/allomorp...>

## 6. Illustrations

You can visualize the generated structures using ASE's GUI (requires a display).

### Degrees of freedom 

The figure below illustrates the degrees of freedom varied to generate the initial conformations of bimetallic nanoparticles.

<p align="center">
  <img src="figs/BNPs_DoF.png" width="500">
</p>

**Description:** Monometallic nanoparticles with different (a) elements, (b) diameters, and (c) shapes are modified or combined to form (d) bimetallic nanoparticles with different atomic orderings.

### Chemical Ordering of Trimetallic Nanoparticles

The figure below illustrates the different chemical orderings in trimetallic nanoparticles.

<p align="center">
  <img src="figs/TNPs_chemical_ordering.jpeg" width="500">
</p>

**Description:** Diagrammatical representations of different compositional descriptors used in multicomponent nanomaterials (left to right) noting ordered structures (o-) where appropriate: bimetallic core@shell (M1@M2), trimetallic inner-core@core@shell (M1@M2@M3), core@random-shell (M1@M2M3), random alloy (M1M2), ordered-core@shell (o-M1M2@M3), core@ordered-shell (M1@o-M2M3), ordered alloy (o-M1M2), random-core@shell (M1M2@M3), core@shell where M2 is distributed across the particle (M1M2@M2M3, often described as an AB@AC structure), ordered alloy with M3 randomly distributed (o-M1M2-M3), random trimetallic solid solution (M1M2M3), and ordered trimetallic alloy or intermetallic solution (o-M1M2M3). Where a support is used, the nomenclature will be given as M1M2M3/support. (Fig. 2 from Crawley et al. Heterogeneous Trimetallic Nanoparticles as Catalysts, Chem. Rev. 2022, 122, 6, 6795-6849)

### Figure 3: Flow diagram of TNP data generation.

<p align="center">
  <img src="figs/TNPs_generation_flowchart.png" width="500">
</p>

**Description:** This diagram illustrates the workflow of generating trimetallic nanoparticle data, starting from monometallic templates, through bimetallic intermediates, to the final trimetallic structures with specific chemical orderings.